# 04 — Decode & durations

GMM has no decode step (`predict` is pointwise). HMM look-ahead is in Viterbi/smoothed only; filtered decode is causal.

In [4]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))
from regime_utils import *

std_variants = {
    "full": load_full_sample_std(),
    "rolling": load_rolling_std(),
    "expanding": load_expanding_std(),
}

rows = []
for std_name, df in std_variants.items():
    X = df.values
    gmm = fit_gmm(X)
    hmm = fit_hmm(X)
    gmm_l = gmm.predict(X)
    vit = viterbi_decode(hmm, X)
    sm = smoothed_decode(hmm, X)
    filt = filtered_decode(hmm, X)
    for method, labels in [
        ("GMM pointwise", gmm_l),
        ("HMM Viterbi", vit),
        ("HMM smoothed", sm),
        ("HMM filtered", filt),
    ]:
        rows.append({
            "Std variant": std_name,
            "Method": method,
            "Mean duration (weeks)": round(run_length_mean(labels), 1),
        })

duration_table = pd.DataFrame(rows)
print(duration_table.to_string(index=False))


Std variant        Method  Mean duration (weeks)
       full GMM pointwise                    9.0
       full   HMM Viterbi                   50.5
       full  HMM smoothed                   44.7
       full  HMM filtered                   25.6
    rolling GMM pointwise                    6.2
    rolling   HMM Viterbi                   18.9
    rolling  HMM smoothed                   17.2
    rolling  HMM filtered                   13.5
  expanding GMM pointwise                    7.3
  expanding   HMM Viterbi                   26.5
  expanding  HMM smoothed                   25.6
  expanding  HMM filtered                   16.4


In [5]:

# Transition entropy (full-sample std, integer labels)
df_full = std_variants["full"]
X = df_full.values
gmm = fit_gmm(X)
hmm = fit_hmm(X)
for name, labels in [
    ("GMM", gmm.predict(X)),
    ("HMM Viterbi", viterbi_decode(hmm, X)),
    ("HMM filtered", filtered_decode(hmm, X)),
]:
    print(f"{name} transition entropy (bits):\n", transition_entropy(labels).round(3))


GMM transition entropy (bits):
 0    0.618
1    0.524
2    0.643
dtype: float64
HMM Viterbi transition entropy (bits):
 0    0.237
1    0.112
2    0.121
dtype: float64
HMM filtered transition entropy (bits):
 0    0.412
1    0.213
2    0.184
dtype: float64


In [6]:

# Cross-model ARI on integer labels (no remapping)
gmm_l = gmm.predict(X)
for decode_name, hmm_l in [("Viterbi", viterbi_decode(hmm, X)), ("filtered", filtered_decode(hmm, X))]:
    print(f"ARI GMM vs HMM {decode_name}: {cross_model_ari(gmm_l, hmm_l):.3f}")

print("\nGMM ~9 wk duration is honest (no L2 decode). HMM Viterbi ~50 wk was inflated by global smoothing.")


ARI GMM vs HMM Viterbi: 0.254
ARI GMM vs HMM filtered: 0.274

GMM ~9 wk duration is honest (no L2 decode). HMM Viterbi ~50 wk was inflated by global smoothing.
